<div style="background:linear-gradient(120deg,#0b1d3a 0%,#4b1f8c 45%,#d6913f 100%);padding:30px 34px;border-radius:16px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
<div style="font-size:13px;letter-spacing:3px;opacity:.85">PLAYGROUND S6E10 · PREDICTING AIRLINE SATISFACTION · ROC AUC</div>
<div style="font-size:36px;font-weight:800;margin:8px 0 6px">🏔️ Summit Composer v2 · 🧅 TabPFN Peel</div>
<div style="font-size:17px;opacity:.95">The public 0.96213 file is a 10% TabPFN layer on top of a 0.96210 base that grew out of <b>this notebook's v1 file</b>.
Peel that layer off exactly, step the base the way it was already moving, put TabPFN back at a better share: <b>public 0.96213</b>, ranked above the public 0.96213 file.</div>
<div style="margin-top:16px;display:inline-block;background:rgba(255,255,255,.18);padding:8px 16px;border-radius:999px;font-weight:700;font-size:18px">Public LB 0.96213</div>
</div>

## 🧭 How the summit was built (every step is a measured public score)

| step | who | move | public |
|---|---|---|---|
| 1 | F.A. Nina, Кирилл Тымчишин | h-blend of four public files + CV-stack "teacher", gated by member disagreement | 0.96206 |
| 2 | KRISH RABARI | straight line `V34 + t·e` from his surface peak | 0.96207 |
| 3 | **me** | Nina's h-blend (7) v10 recipe on rabari's line point V43 | **0.96208** |
| 4 | KRISH RABARI | extend that composition: bolder line point, teacher share, weights (V54) | 0.96209 |
| 5 | **me** | V54 + reversed disagreement gate; V54's bold move ×1.5 (**v1 of this notebook**) | **0.96210** |
| 6 | F.A. Nina | a small extra step on my v1 file (her ps-s6e10 notebook, now private) | 0.96210 |
| 7 | Mehran Kazeminia, Yuvraj Lamba | `pct(0.10 · TabPFN + 0.90 · step 6)`: a real, non-tree model as a 10% layer | 0.96213 |
| 8 | **me** | peel step 7 apart, step 6 ×5, TabPFN share 0.08 (**this version**) | **0.96213**, ranked above step 7 |

Step 7 is the first gain since 0.96206 that comes from a **different model**, not from re-weighting the same public files.

## ⚠️ Before you pick this as a final submission

- Every move here was chosen on the **public 20%** of rows. Public gains of 0.00001 are inside public noise (sd ≈ 0.0012·√(1−ρ), Korokke3).
- In my CV test, adding 5–15% of a foundation model to a strong 181-member CV stack **lowered** OOF AUC. The TabPFN gain is public-LB evidence only.
- Keep one **CV-backed** stack as your other final pick (for example `own_cv_stack_181_test.npy` in my atlas dataset, 5-fold CV 0.962166).

## ⚙️ Settings: change these to make it yours

Defaults reproduce the scored **0.96213** file exactly.

In [ ]:
# rabari's V54 recipe (the base, fixed): t=12, teacher share 0.45, CV-stack share -1,
# member weights (-0.25, 0.10, 0.15, 1.00), corrections (0.14, -0.02, -0.04, -0.08), no gate.
V54_RECIPE = dict(t=12.0, teacher_w=0.45, own_share=-1.0, main=[-0.25, 0.10, 0.15, 1.00], corr=[0.14, -0.02, -0.04, -0.08], gate_amp=0.0)

# ---- this notebook's move (defaults = V54's bold move x1.5 + reversed gate, public 0.96210) ----
T = 13.0                                   # point on rabari's line V34 + t*e            (V54: 12)
TEACHER_W = 0.58                           # average teacher share in each member        (V54: 0.45)
OWN_SHARE = -1.5                           # teacher = (1 - OWN_SHARE)*T188 + OWN_SHARE*CV_stack   (V54: -1)
MAIN_W = [-0.35, 0.10, 0.15, 1.10]         # member weights (0.96201, 0.96203, 0.96204, line point)
ORDER_C = [0.175, -0.025, -0.05, -0.10]    # position corrections after sorting each row
GATE_AMP = -0.05                           # reversed disagreement gate: share = TEACHER_W + GATE_AMP*(pct(member std) - 0.5)
# Set everything to the V54 values and GATE_AMP = 0 to get V54 back; keep V54 values with GATE_AMP = -0.05 for the plain gate file (also 0.96210).

# ---- v2: the TabPFN layer (defaults = public 0.96213, this notebook's scored file) ----
PUBLIC_TAB_W = 0.10   # TabPFN share inside the public 0.96213 file (Mehran / Yuvraj recipe); used to peel it
NINA_STEP = 5.0       # base = P2 + NINA_STEP * (S - P2); 1.0 = Nina's base as it is
TAB_W = 0.08          # TabPFN share when re-mixing (public file: 0.10)
# NINA_STEP = 1.0 with TAB_W = 0.10 gives the public 0.96213 file back exactly.

## 📦 1. Load the inputs (all hash-pinned)

In [ ]:
import os, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
warnings.filterwarnings('ignore')

ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path(os.environ['LOCAL_KAGGLE_INPUT'])

def find(name, hint=''):
    hits = sorted(p for p in ROOT.rglob(name) if hint in str(p))
    assert hits, f'{name} ({hint}) not found'
    return hits[0]

ids = pd.read_csv(find('sample_submission.csv', 'playground-series-s6e10')).id.to_numpy()
N = len(ids)
pct = lambda v: (rankdata(np.asarray(v, dtype=np.float64), method='average') - 0.5) / len(v)

PINNED = {  # (file, input slug): sha256
    ('0.96206.csv', 'ps-s6e10-7'): 'de8438a811025afe781ff32af09ae0ada7a035741cdb98c1029be06ddba6143f',
    ('0.96188.txt', 'ps-s6e10-7'): '95104146ff66aac2d61462e472c0c9e3097b8403d085b10bf4bc24c84d40fd58',
    ('0.96201.csv', 'ps-s6e10-7'): '8883773ff30ba9a71581657b9daf881ac45652031b6e3ee565844dbcff35797a',
    ('0.96203.csv', 'ps-s6e10-7'): 'b31119ef653e50b71452187174d6c85d7929550c23b4b020797f19f5f132112d',
    ('0.96204.csv', 'ps-s6e10-7'): 'c7d3b9e4c1c377c74e0f75b475c50319413a264fc4401c0a00fe689a99542873',
    ('rabari_v41_submission.csv', 's6e10-lb-surface-atlas'): '7d0e2547eebde1746f0a0fa40d427cf5f944617514e7b022499901f8b3426265',
    ('rabari_v43_submission.csv', 's6e10-lb-surface-atlas'): '193d53924035c155256c7b85bb9c42e51fa3bd69c25db7a9d9bd7ee0451ca55a',
    ('rabari_v54_submission.csv', 's6e10-lb-surface-atlas'): 'dec9cf59323138bddad66c7d2e8d898877a67a1acf5c664b7e8b5af53700a9fc',
    ('own_cv_stack_181_test.npy', 's6e10-lb-surface-atlas'): '1f2fc66c782a8e1175db33318b272d1194b71d22a98fbbbfb54a1814a1b31b39',
}
for (f, hint), h in PINNED.items():
    assert hashlib.sha256(find(f, hint).read_bytes()).hexdigest() == h, f'{f} changed upstream'

def raw(name, hint):
    p = find(name, hint)
    if p.suffix == '.npy':
        v = np.load(p).astype(np.float64)
    else:
        d = pd.read_csv(p); d.columns = ['id', 'p']
        v = d.set_index('id').p.reindex(ids).to_numpy(np.float64)
    assert v.shape == (N,) and np.isfinite(v).all(), name
    return v

RC = pct(raw('0.96206.csv', 'ps-s6e10-7'))                       # public 0.96206 (Kirill's gate)
T188 = raw('0.96188.txt', 'ps-s6e10-7')                          # ewtherka's plain-LR stack
BASE3 = [raw(f, 'ps-s6e10-7') for f in ('0.96201.csv', '0.96203.csv', '0.96204.csv')]
V41 = pct(raw('rabari_v41_submission.csv', 's6e10-lb-surface-atlas'))   # rabari line, t = 1.25
V43 = pct(raw('rabari_v43_submission.csv', 's6e10-lb-surface-atlas'))   # rabari line, t = 4.5
V54 = pct(raw('rabari_v54_submission.csv', 's6e10-lb-surface-atlas'))   # rabari V54, public 0.96209
OWN = raw('own_cv_stack_181_test.npy', 's6e10-lb-surface-atlas')        # my 181-member CV stack
print('✅ all 9 inputs hash-verified')

In [ ]:
PINNED_V2 = {
    ('submission.csv', '1-airline-s6e10-tabpfn'): 'ff52a2fe98a3d41f40e9c40421b31abc46e6993a0211bb87bc07be30faf6f886',
    ('submission.csv', 'airline-passenger-satisfaction-0-96213'): '9dcab522f25f2f1603415bab644b9523e2c55b424a7e9ffaa65fa3ecceeaff5e',
}
for (f, hint), h in PINNED_V2.items():
    assert hashlib.sha256(find(f, hint).read_bytes()).hexdigest() == h, f'{hint}/{f} changed upstream'
TABPFN = raw('submission.csv', '1-airline-s6e10-tabpfn')                        # Mehran's TabPFN, public 0.96100 (probabilities)
PUBLIC213 = pct(raw('submission.csv', 'airline-passenger-satisfaction-0-96213'))  # public 0.96213 = pct(0.10*TabPFN + 0.90*Nina 0.96210)
print('✅ TabPFN and the public 0.96213 file hash-verified')

## 🧩 2. Helpers: Nina's h-blend, rabari's line, the gated composition, the TabPFN peel

In [ ]:
%%writefile summit_composer.py
"""Summit Composer helpers. Reuse: add this notebook as an input, then `from summit_composer import *`."""
import numpy as np
from scipy.stats import rankdata


def pct(v):
    return (rankdata(np.asarray(v, dtype=np.float64), method='average') - 0.5) / len(v)


def hblend(M, main, corr, mix=(0.3, 0.7)):
    """F.A. Nina's h-blend: sort each row both ways, weight by member + position, mix the two sides (float32 like the original)."""
    n, sides = len(M), []
    for rev in (False, True):
        order = np.argsort(-M if rev else M, axis=1, kind='stable')
        s = np.zeros(n)
        for j in range(M.shape[1]):
            k = order[:, j]
            s += M[np.arange(n), k] * (main[k] + corr[j])
        sides.append(s.astype(np.float32))
    return (np.float32(mix[0]) * sides[0] + np.float32(mix[1]) * sides[1]).astype(np.float32)


def rabari_line(t, RC, V41, V43):
    """KRISH RABARI's line V34 + t*e, rebuilt from his measured points t = 1.25 (V41) and t = 4.5 (V43)."""
    a125, a450 = V41 - RC, V43 - RC
    return pct(RC + a125 + (t - 1.25) / 3.25 * (a450 - a125))


def compose(base3, fourth, teacher, teacher_w, main, corr, gate_amp=0.0):
    """Mix each member with the teacher (share gated by member disagreement), then h-blend."""
    M = np.column_stack(list(base3) + [fourth])
    w = teacher_w + gate_amp * (pct(M.std(axis=1)) - 0.5)
    return pct(hblend(M * (1 - w[:, None]) + w[:, None] * teacher[:, None], np.asarray(main, float), np.asarray(corr, float)))


def peel(public, layer, w, ref):
    """Undo public = pct(w*layer + (1-w)*S) for S (raw scale). The quantiles of the mix come from `ref`, a base close to S."""
    q = np.sort(w * layer + (1 - w) * ref)
    k = np.rint(public * len(q) + 0.5).astype(int) - 1
    return (q[k] - w * layer) / (1 - w)


def layer(base, model_prob, w):
    """Mehran's TabPFN layer: mix the model's probabilities into a rank base, then re-rank."""
    return pct(w * model_prob + (1 - w) * base)

## 🏗️ 3. Rebuild v1's file (V54 + bold move ×1.5 + reversed gate, public 0.96210)

In [ ]:
from summit_composer import pct, hblend, rabari_line, compose

def recipe(t, teacher_w, own_share, main, corr, gate_amp):
    teacher = (1 - own_share) * T188 + own_share * OWN
    return compose(BASE3, rabari_line(t, RC, V41, V43), teacher, teacher_w, main, corr, gate_amp)

base = recipe(**V54_RECIPE)
moved = recipe(T, TEACHER_W, OWN_SHARE, MAIN_W, ORDER_C, GATE_AMP)
fourth = rabari_line(T, RC, V41, V43)

# Check: without the gate the recipe rebuilds rabari's public V54 (his direction files are private,
# so the line is rebuilt from his V41/V43 outputs; a handful of rows can differ in the last digit).
print(f'recipe vs rabari V54: rank corr {np.corrcoef(base, V54)[0, 1]:.10f} | rows with a different rank: {(base != V54).sum()}')
assert np.corrcoef(base, V54)[0, 1] > 0.99999999

# Apply only the change this notebook's move makes, on top of the real V54 file.
final = pct(V54 + (moved - base))
print(f'final vs V54: rank corr {np.corrcoef(final, V54)[0, 1]:.7f} | vs 0.96206: {np.corrcoef(final, RC)[0, 1]:.7f}')
P2 = final   # v1's file (public 0.96210)

## 🧅 The new lever: peel the TabPFN layer off the public file

The public 0.96213 file is `R = pct(0.10·TabPFN + 0.90·S)`, where `S` is Nina's 0.96210 file. Her notebook is private now, so `S` is not available.
But `pct` only throws away the **scale**, not the order, so `S` can be recovered:

1. The mixed values `m = 0.10·TabPFN + 0.90·S` have almost the same distribution for any base close to `S`. My v1 file `P2` is very close (rank corr 0.9999994), so the sorted values of `0.10·TabPFN + 0.90·P2` are a very good estimate of the quantiles of `m`.
2. Row `i` has rank `R_i`, so `m_i ≈ quantile(R_i)`.
3. Solve for the base: `S_i = (m_i − 0.10·TabPFN_i) / 0.90`.

Check on a known case: peeling `pct(0.10·TabPFN + 0.90·P2)` gives `P2` back with rank error sd 3·10⁻⁵ (a tenth of a rank step). Peeling the public file and re-mixing at 0.10 gives the public file back **exactly**.

With `S` in hand, two separate moves become possible. I measured each one before combining them:

| file | TabPFN share | base | public |
|---|---|---|---|
| public file (Mehran / Yuvraj) | 0.10 | Nina `S` | 0.96213 |
| my v1 gate file + TabPFN | 0.10 | V54 + gate | 0.96212 |
| same | 0.15 | V54 + gate | 0.96211 (too much TabPFN) |
| **peeled, share 0.08** | 0.08 | Nina `S` | **0.96213**, my team jumped from rank 69 to 15 |
| **peeled, Nina's step ×5** | 0.10 | `P2 + 5·(S − P2)` | **0.96213** |
| **both (this notebook)** | 0.08 | `P2 + 5·(S − P2)` | **0.96213** |

Equal displayed scores are ordered by the unrounded score, then by the **earlier** submission. My files came later than the public 0.96213 file, yet the rank went from 69 to 15, ahead of the teams that hold that file.
So the 0.08 / ×5 moves beat it on the unrounded public score.

Things that did **not** work (all one probe each, on top of the public 0.96213 file): +5% of kof3st's TabPFN-3.5 (0.96212), +5% of Mehran's LGBM-XGB (0.96211), my bold move ×2 (0.96212).
In CV, a disagreement gate on the TabPFN share lost on every base I tried, so the share stays flat.

## 🧅 4. Peel, step, re-mix

In [ ]:
from summit_composer import peel, layer

S_raw = peel(PUBLIC213, TABPFN, PUBLIC_TAB_W, P2)              # Nina's hidden 0.96210 base
assert (layer(S_raw, TABPFN, PUBLIC_TAB_W) == PUBLIC213).all()  # re-mixing gives the public file back exactly
S = pct(S_raw)
print(f'peeled base vs my v1 file: rank corr {np.corrcoef(S, P2)[0, 1]:.9f}')

based = pct(S + (NINA_STEP - 1) * (S - P2))                     # step Nina's move on my v1 file
final = layer(based, TABPFN, TAB_W)
print(f'final vs public 0.96213: rank corr {np.corrcoef(final, PUBLIC213)[0, 1]:.8f} | vs v1 file: {np.corrcoef(final, P2)[0, 1]:.7f}')

## 🔍 5. Check and look

In [ ]:
DEFAULTS = (T, TEACHER_W, OWN_SHARE, MAIN_W, ORDER_C, GATE_AMP, PUBLIC_TAB_W, NINA_STEP, TAB_W) == \
    (13.0, 0.58, -1.5, [-0.35, 0.10, 0.15, 1.10], [0.175, -0.025, -0.05, -0.10], -0.05, 0.10, 5.0, 0.08)
if DEFAULTS:
    assert hashlib.sha256(final.astype('<f8').tobytes()).hexdigest() == '21f6fe37dd8f1b7673677d00d6d1bbd75190534d0d557dee76b9fd897a4fdfed'
    print('✅ identical to my scored 0.96213 file')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
mid = (P2 > 0.2) & (P2 < 0.8)
ax[0].scatter(P2[mid][::40], (final - P2)[mid][::40], s=2, alpha=.4, c='#4b1f8c')
ax[0].set_title('Rank change vs the v1 file (middle 60% of rows)', fontweight='bold')
ax[0].set_xlabel('v1 file rank'); ax[0].set_ylabel('final − v1')
labels = ['P1 +.10', 'P1 +.15', 'pub +kof .05', 'pub +lgbxgb .05', 'bold×2 +.10', 'public file', 'peel .08', 'peel ×5', 'both']
ax[1].plot(labels, [0.96212, 0.96211, 0.96212, 0.96211, 0.96212, 0.96213, 0.96213, 0.96213, 0.96213], 'o-', c='#d6913f', lw=2, ms=9)
ax[1].tick_params(axis='x', rotation=35)
ax[1].set_title('Measured public score, round 10', fontweight='bold'); ax[1].set_ylabel('public LB (rounded)')
plt.tight_layout(); plt.show()

## 📤 6. Write the submission

In [ ]:
sub = pd.DataFrame({'id': ids, 'satisfaction': final})
assert sub.satisfaction.between(0, 1).all() and sub.id.is_unique and len(sub) == N
sub.to_csv('submission.csv', index=False, float_format='%.9f')
print('exported:', sorted(p for p in os.listdir('.') if p.endswith(('.csv', '.py'))))
sub.head()

## 🔁 Make it yours

**1. Peel any public "model layer" file.** If a public file is `pct(w·model + (1−w)·base)` and you have the model's output, `peel(public, model, w, ref)` gives the base back. `ref` only needs to be close to the base (rank corr ≳ 0.9999).

**2. Re-mix at your own share.** `layer(base, model_prob, w)` is the TabPFN layer. Change `TAB_W` and `NINA_STEP` in the settings cell; `NINA_STEP = 1.0, TAB_W = 0.10` gives the public 0.96213 file back.

**3. Bring your own model.** Any model with real, different signal (a TFM, a new feature set) can be the layer. Probability scale beat rank scale in my CV test, so mix probabilities, as Mehran did.

**4. Reuse the pieces.** Add this notebook as an input:

```python
import sys; sys.path.append('/kaggle/input/<this-notebook>')
from summit_composer import peel, layer, compose, rabari_line, hblend, pct
```

My 181-member CV stack (test **and** OOF, CV 0.962166) and the rehosted rabari files are in
[s6e10-lb-surface-atlas](https://www.kaggle.com/datasets/mayankraaz/s6e10-lb-surface-atlas).

## 🙏 Credits

This notebook composes other people's measured work. Please upvote them:
- **[Mehran Kazeminia](https://www.kaggle.com/code/mehrankazeminia/1-airline-s6e10-tabpfn)**: the TabPFN model (11 hours of compute) used as the layer, and the [10% TabPFN ensembling idea](https://www.kaggle.com/code/mehrankazeminia/3-airline-s6e10-ensembling) behind the public 0.96213.
- **[Yuvraj Lamba](https://www.kaggle.com/code/yuvrajlamba01/airline-passenger-satisfaction-0-96213)**: the public 0.96213 file that this notebook peels (input, used unchanged).
- **[F.A. Nina](https://www.kaggle.com/nina2025)**: the h-blend rule, the [ps-s6e10-7](https://www.kaggle.com/datasets/nina2025/ps-s6e10-7) dataset, and the 0.96210 step under the public 0.96213 file.
- **[KRISH RABARI](https://www.kaggle.com/code/rabari9999/s6e10-lb-0-96209-rank-10)**: the surface line and the V54 recipe. His V41, V43 and V54 outputs are rehosted unchanged in my atlas dataset.
- **[Кирилл Тымчишин](https://www.kaggle.com/code/kirilltym/0-96206-disagreement-gated-teachers)**: the disagreement gate that v1 reverses, and the 0.96206 anchor.
- **[ewtherka](https://www.kaggle.com/code/ewtherka/s6e10-0-96188-lb-l2-family-wise-actually-helps)**: the 0.96188 plain-LR stack used as the teacher.
- **daoviet** and **Prabhav Semwal (yeashusemwal)** for their write-ups of the TabPFN blend; **kof3st** for the TabPFN-3.5 file I tested; **Chris Deotte** and every public OOF author for the stacks underneath.
- **[Korokke3](https://www.kaggle.com/code/korokke3/s6e10-is-your-0-00001-real-lb-noise-calculator)**: the noise formula in the warning.

If the peel trick was useful, an upvote helps. Thank you! ✈️